# Gold Layer

## Load cleaned transactions from Silver layer

In [0]:
silver_df = spark.read.table("retail_silver_cleaned")

In [0]:
silver_df.display()

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,email,city,registration_date,full_name,product_name,category,price,store_name,location,total_amount
5,7,101,28,3,2024-11-15,user101@example.com,Delhi,2023-09-14,Ravi Yadav,Smartwatch,Electronics,4999.0,Mega Plaza,Chennai,14997.0
3,1,102,11,2,2024-08-11,user102@example.com,Mumbai,2024-01-21,Nina Joshi,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,1599.98
4,1,103,18,3,2024-09-05,user103@example.com,Bangalore,2023-07-10,Sonal Sharma,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.97
3,3,104,13,4,2025-05-04,user104@example.com,Hyderabad,2024-02-05,Karan Patel,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
3,1,105,21,5,2024-10-02,user105@example.com,Chennai,2023-06-28,Riya Singh,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,3999.95
2,5,105,5,1,2025-03-17,user105@example.com,Chennai,2023-06-28,Riya Singh,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
4,3,105,2,5,2024-11-12,user105@example.com,Chennai,2023-06-28,Riya Singh,Yoga Mat,Fitness,499.0,Downtown Mini Store,Pune,2495.0
3,9,107,22,4,2024-11-16,user107@example.com,Ahmedabad,2023-05-12,Priya Kapoor,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
1,5,108,12,4,2025-05-26,user108@example.com,Kolkata,2023-08-19,Rahul Verma,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
5,8,109,17,5,2024-07-10,user109@example.com,Delhi,2024-04-01,Pooja Mehta,Desk Organizer,Accessories,399.0,Mega Plaza,Chennai,1995.0


In [0]:
from pyspark.sql.functions import sum, countDistinct, avg

gold_df = silver_df.groupBy(
    "transaction_date",
    "product_id", "product_name", "category",
    "store_id", "store_name", "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)

In [0]:
gold_df.display()

transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2024-07-14,1,Wireless Mouse,Electronics,5,Mega Plaza,Chennai,1,799.99,1,799.99
2024-07-17,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,5,3999.95,1,3999.95
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.97,1,2399.97
2025-06-03,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45


In [0]:
gold_df.write.mode("overwrite").format("delta").saveAsTable("retail_gold_sales_summary")


In [0]:
%sql
select * from retail_gold_sales_summary

transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2024-07-14,1,Wireless Mouse,Electronics,5,Mega Plaza,Chennai,1,799.99,1,799.99
2024-07-17,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,5,3999.95,1,3999.95
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.97,1,2399.97
2025-06-03,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
